In [3]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
import anndata as ad
import scanpy as sc
from pathlib import Path
from tqdm import tqdm
tqdm.pandas()

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica']
plt.rcParams['font.size'] = 12
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['pdf.fonttype'] = 42

In [5]:
COUNTS_H5AD = Path('/data1/normantm/eli/T7/202511_RPE1_dg50/analysis/intermediate_files/251204_dg50_gex.h5ad')
MODEL_DIR = Path("/data1/normantm/eli/T7/202511_RPE1_dg50/analysis/intermediate_files/dg50_scvi")
OUT_DIR = Path("/data1/normantm/eli/T7/202511_RPE1_dg50/analysis/intermediate_files/scvi_direct_emb")
N_SPLITS = 4

adata = sc.read('/data1/normantm/eli/T7/202511_RPE1_dg50/analysis/intermediate_files/251204_dg50_gex.h5ad')
latent = adata.obsm["X_scvi"]
conditions = adata.obs["guide_target"].astype(str).to_numpy()

condition_levels = pd.Index(sorted(pd.unique(conditions)))
condition_codes = pd.Categorical(conditions, categories=condition_levels).codes
cell_hash = pd.util.hash_pandas_object(
    pd.Index(adata.obs_names), index=False
).to_numpy(np.uint64)
ordering = np.lexsort((cell_hash, condition_codes))
ordered_codes = condition_codes[ordering]
new_condition = np.r_[True, ordered_codes[1:] != ordered_codes[:-1]]
condition_start = np.maximum.accumulate(
    np.where(new_condition, np.arange(adata.n_obs), 0)
)
rank_within_condition = np.arange(adata.n_obs) - condition_start
split_id = np.empty(adata.n_obs, dtype=np.int8)
split_id[ordering] = rank_within_condition % N_SPLITS

columns = [f"latent_{i + 1:02d}" for i in range(latent.shape[1])]

def condition_means(mask):
    frame = pd.DataFrame(latent[mask], columns=columns)
    frame["condition"] = conditions[mask]
    means = frame.groupby("condition", sort=True).mean()
    means = means.subtract(means.loc["NTC"], axis="columns")
    return means.drop(index="NTC")

OUT_DIR.mkdir(parents=True, exist_ok=True)
condition_means(np.ones(adata.n_obs, dtype=bool)).to_csv(
    "/data1/normantm/eli/T7/202511_RPE1_dg50/analysis/intermediate_files/scvi_direct_emb/scvi_full_ntc_subtracted.csv",
    index_label="condition",
)

for i in range(N_SPLITS):
    condition_means(split_id == i).to_csv(
        OUT_DIR / f"/data1/normantm/eli/T7/202511_RPE1_dg50/analysis/intermediate_files/scvi_direct_emb/scvi_split{i}_ntc_subtracted.csv",
        index_label="condition",
    )

In [7]:
# whitening

DATA = Path('/data1/normantm/eli/T7/202511_RPE1_dg50/analysis/intermediate_files/scvi_direct_emb')

full = pd.read_csv(DATA / "scvi_full_ntc_subtracted.csv", index_col=0)
splits = [
    pd.read_csv(DATA / f"scvi_split{i}_ntc_subtracted.csv", index_col=0)
    for i in range(4)
]

# No centering: every row is already an effect relative to NTC.
values = full.to_numpy(float)
_, singular_values, right_vectors = np.linalg.svd(
    values, full_matrices=False
)
rotation = right_vectors.T
observed_variance = singular_values**2 / len(full)

split_scores = np.stack(
    [split.to_numpy(float) @ rotation for split in splits], axis=1
)
sampling_noise = split_scores.var(axis=1, ddof=1).mean(axis=0) / 4

noise_floor = np.median(sampling_noise)
denominator_variance = np.maximum(observed_variance, noise_floor)
whitened = values @ rotation / np.sqrt(denominator_variance)

columns = [f"whitened_component_{i:02d}" for i in range(1, 65)]
whitened = pd.DataFrame(whitened, index=full.index, columns=columns)
whitened.to_csv(
    DATA / "scvi_recomputed_whitened_components.csv",
    index_label="condition",
)
print(
    f"noise floor = {noise_floor:.10g};",
    f"components above floor = {(observed_variance > noise_floor).sum()}",
)

noise floor = 3.960378211e-08; components above floor = 40


In [8]:
def get_geometry_nd(df_singles, df_doubles, ab):
    """
    N-dimensional generalization of get_geometry's plane-distance decomposition.
    va, vb, v_ab are 1D arrays of the same (arbitrary) dimension.
    """

    a, b = ab.split('_')
    va = np.array(df_singles.loc[a])
    vb = np.array(df_singles.loc[b])
    v_ab = np.array(df_doubles.loc[ab])

    A = np.stack([va, vb], axis=1)          # (n_dims, 2)
    # orthonormal basis for span(va, vb) via QR
    Q, _ = np.linalg.qr(A)                   # Q: (n_dims, 2), orthonormal columns

    v_proj = Q @ (Q.T @ v_ab)                # projection of v_ab onto span(va, vb)
    v_orth = v_ab - v_proj                   # component orthogonal to the plane

    r_neo = np.linalg.norm(v_orth)           # generalized out-of-plane distance
    r_mixed = np.linalg.norm(v_proj)         # in-plane component magnitude (analogous to your r_mixed derivation)

    additive_singles = va + vb
    sim_additive = np.dot(v_ab, additive_singles) / (np.linalg.norm(v_ab) * np.linalg.norm(additive_singles))
    sim_a = np.dot(v_ab, va) / (np.linalg.norm(v_ab) * np.linalg.norm(va))
    sim_b = np.dot(v_ab, vb) / (np.linalg.norm(v_ab) * np.linalg.norm(vb))

    gamma = np.dot(v_proj, additive_singles) / (np.linalg.norm(v_proj) * np.linalg.norm(additive_singles))

    r_additive = np.linalg.norm(additive_singles)
    r_syn = r_mixed - r_additive

    return {'metrics': (r_neo, r_mixed, r_syn, r_additive, sim_additive, sim_a, sim_b, gamma), 'additive_component': v_proj}

In [20]:
df_scvi = pd.DataFrame(whitened, index=full.index, columns=columns)
df_singles, df_doubles = df_scvi.query("not index.str.contains('_')"), df_scvi.query("index.str.contains('_')")
geometry = pd.DataFrame(index = df_doubles.index, columns = ["r_neo_raw", "r_mixed", "r_syn_raw", "r_additive", "sim_additive", "sim_a", "sim_b", "gamma"])
additive_components = pd.DataFrame(index = df_doubles.index, columns = list(range(64)))
for i in df_doubles.index:
    results_dict = get_geometry_nd(df_singles, df_doubles, i)
    geometry.loc[i] = results_dict['metrics']
    additive_components.loc[i] = results_dict['additive_component']
geometry = geometry.astype(np.float64)
geometry['dominant_tgt'] = geometry.progress_apply(lambda df: df.name.split('_')[0] if df.sim_a > df.sim_b else df.name.split('_')[1], axis = 1)

100%|██████████| 1225/1225 [00:00<00:00, 88046.17it/s]


In [21]:
df_norm = additive_components.apply(np.linalg.norm, axis = 1).to_frame('norm')
df_doubles = df_norm.query("index.str.contains('_')").copy()
df_doubles['norm_additive'] = df_doubles.index.map(lambda i: np.linalg.norm(df_scvi.loc[i.split('_')[0]] + df_scvi.loc[i.split('_')[1]]))

from sklearn.linear_model import LinearRegression
model = LinearRegression(fit_intercept=False)
model.fit(df_doubles[['norm_additive']], df_doubles['norm'])
additive_coef = model.coef_[0]
print(additive_coef)
print(model.score(df_doubles[['norm_additive']], df_doubles['norm']))

geometry['r_syn'] = geometry.r_mixed - additive_coef * geometry.r_additive

0.6658188270481328
0.39498619846622396


In [22]:
ncells_dict = adata.obs.guide_target.value_counts().to_dict()
ncells = pd.DataFrame(index = df_doubles.index)
ncells['single_a'] = ncells.index.map(lambda x: x.split('_')[0])
ncells['single_b'] = ncells.index.map(lambda x: x.split('_')[1])
ncells['n_double'] = ncells.index.map(ncells_dict)
ncells['n_single_a'] = ncells.single_a.map(ncells_dict)
ncells['n_single_b'] = ncells.single_b.map(ncells_dict)
ncells['harmonic_single_n'] = 2 * ncells.n_single_a * ncells.n_single_b / (ncells.n_single_a + ncells.n_single_b)

import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
from sklearn.preprocessing import StandardScaler

def weighted_quantile(values, weights, quantile):
    order = np.argsort(values)
    values = np.asarray(values)[order]
    weights = np.asarray(weights)[order]
    cumulative = np.cumsum(weights) - 0.5 * weights
    cumulative /= weights.sum()
    return np.interp(quantile, cumulative, values)


def correct_rneo(df_scores, df_cells, score_col="r_neo_raw",
                 n_neighbors=100):

    df = df_scores[[score_col]].join(
        df_cells[[
            "single_a",
            "single_b",
            "n_double",
            "n_single_a",
            "n_single_b",
            "harmonic_single_n",
        ]],
        how="inner",
    )

    count_features = pd.DataFrame(
        {
            "log_double_n": np.log1p(df["n_double"]),
            "log_harmonic_single_n": np.log1p(df["harmonic_single_n"]),
            "single_count_imbalance": np.abs(
                np.log(df["n_single_a"]) - np.log(df["n_single_b"])
            ),
        },
        index=df.index,
    )

    X = StandardScaler().fit_transform(count_features)
    distances = cdist(X, X)

    a = df["single_a"].to_numpy()
    b = df["single_b"].to_numpy()

    eligible = (
        (a[:, None] != a[None, :])
        & (a[:, None] != b[None, :])
        & (b[:, None] != a[None, :])
        & (b[:, None] != b[None, :])
    )

    raw = df[score_col].to_numpy(float)
    log_raw = np.log(np.maximum(raw, np.finfo(float).eps))

    centers = np.empty(len(raw))
    scales = np.empty(len(raw))

    for i in range(len(raw)):
        candidates = np.flatnonzero(eligible[i])
        nearest = candidates[
            np.argsort(distances[i, candidates])[:n_neighbors]
        ]

        bandwidth = max(distances[i, nearest].max(), 1e-8)
        weights = np.exp(
            -0.5 * (distances[i, nearest] / bandwidth) ** 2
        )

        center = weighted_quantile(
            log_raw[nearest],
            weights,
            0.5,
        )

        mad = weighted_quantile(
            np.abs(log_raw[nearest] - center),
            weights,
            0.5,
        )

        centers[i] = center
        scales[i] = max(1.4826 * mad, 1e-8)

    return pd.Series(
        (log_raw - centers) / scales,
        index=df.index,
        name="corrected_rneo",
    )

geometry['r_neo'] = correct_rneo(geometry, ncells)

In [ ]:
geometry.drop(['r_neo_raw', 'r_syn_raw'], axis = 1).to_csv(OUT_DIR / "scvi_direct_global_interaction_metrics.csv")